### RAG Pipelines - Data Ingestion to Vectore DB Pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\Negah\Desktop\THSI\implementation 2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: RAG_KG1. draft1.pdf
  ✓ Loaded 13 pages

Processing: RAG_KG1.pdf
  ✓ Loaded 13 pages

Processing: RAG_KG2.pdf
  ✓ Loaded 5 pages

Total documents loaded: 31


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'iOS Version 18.1.1 (Build 22B91) Quartz PDFContext', 'creator': '', 'creationdate': "D:20251204083830Z00'00'", 'source': '..\\data\\text_files\\pdf\\RAG_KG1. draft1.pdf', 'file_path': '..\\data\\text_files\\pdf\\RAG_KG1. draft1.pdf', 'total_pages': 13, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20251204083830Z00'00'", 'trapped': '', 'modDate': "D:20251204083830Z00'00'", 'creationDate': "D:20251204083830Z00'00'", 'page': 0, 'source_file': 'RAG_KG1. draft1.pdf', 'file_type': 'pdf'}, page_content='Knowledge Graph-Guided Retrieval Augmented Generation\nXiangrong Zhu→\nYuexiang Xie↑\nYi Liu→\nYaliang Li↑\nWei Hu→\n→State Key Laboratory for Novel Software Technology, Nanjing University, China\n↑Alibaba Group\n{xrzhu, yiliu07}.nju@gmail.com, whu@nju.edu.cn\n{yuexiang.xyx, yaliang.li}@alibaba-inc.com\nAbstract\nRetrieval-augmented generation (RAG) has\nemerged as a promising technology for address-\ning hallucin

In [6]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)
chunks


Split 31 documents into 168 chunks

Example chunk:
Content: Knowledge Graph-Guided Retrieval Augmented Generation
Xiangrong Zhu→
Yuexiang Xie↑
Yi Liu→
Yaliang Li↑
Wei Hu→
→State Key Laboratory for Novel Software Technology, Nanjing University, China
↑Alibaba G...
Metadata: {'producer': 'iOS Version 18.1.1 (Build 22B91) Quartz PDFContext', 'creator': '', 'creationdate': "D:20251204083830Z00'00'", 'source': '..\\data\\text_files\\pdf\\RAG_KG1. draft1.pdf', 'file_path': '..\\data\\text_files\\pdf\\RAG_KG1. draft1.pdf', 'total_pages': 13, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20251204083830Z00'00'", 'trapped': '', 'modDate': "D:20251204083830Z00'00'", 'creationDate': "D:20251204083830Z00'00'", 'page': 0, 'source_file': 'RAG_KG1. draft1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'iOS Version 18.1.1 (Build 22B91) Quartz PDFContext', 'creator': '', 'creationdate': "D:20251204083830Z00'00'", 'source': '..\\data\\text_files\\pdf\\RAG_KG1. draft1.pdf', 'file_path': '..\\data\\text_files\\pdf\\RAG_KG1. draft1.pdf', 'total_pages': 13, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20251204083830Z00'00'", 'trapped': '', 'modDate': "D:20251204083830Z00'00'", 'creationDate': "D:20251204083830Z00'00'", 'page': 0, 'source_file': 'RAG_KG1. draft1.pdf', 'file_type': 'pdf'}, page_content='Knowledge Graph-Guided Retrieval Augmented Generation\nXiangrong Zhu→\nYuexiang Xie↑\nYi Liu→\nYaliang Li↑\nWei Hu→\n→State Key Laboratory for Novel Software Technology, Nanjing University, China\n↑Alibaba Group\n{xrzhu, yiliu07}.nju@gmail.com, whu@nju.edu.cn\n{yuexiang.xyx, yaliang.li}@alibaba-inc.com\nAbstract\nRetrieval-augmented generation (RAG) has\nemerged as a promising technology for address-\ning hallucin

### Embedding and VectorStoreDB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


### VectorStore

In [10]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 336


In [11]:
chunks

[Document(metadata={'producer': 'iOS Version 18.1.1 (Build 22B91) Quartz PDFContext', 'creator': '', 'creationdate': "D:20251204083830Z00'00'", 'source': '..\\data\\text_files\\pdf\\RAG_KG1. draft1.pdf', 'file_path': '..\\data\\text_files\\pdf\\RAG_KG1. draft1.pdf', 'total_pages': 13, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20251204083830Z00'00'", 'trapped': '', 'modDate': "D:20251204083830Z00'00'", 'creationDate': "D:20251204083830Z00'00'", 'page': 0, 'source_file': 'RAG_KG1. draft1.pdf', 'file_type': 'pdf'}, page_content='Knowledge Graph-Guided Retrieval Augmented Generation\nXiangrong Zhu→\nYuexiang Xie↑\nYi Liu→\nYaliang Li↑\nWei Hu→\n→State Key Laboratory for Novel Software Technology, Nanjing University, China\n↑Alibaba Group\n{xrzhu, yiliu07}.nju@gmail.com, whu@nju.edu.cn\n{yuexiang.xyx, yaliang.li}@alibaba-inc.com\nAbstract\nRetrieval-augmented generation (RAG) has\nemerged as a promising technology for address-\ning hallucin

In [12]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector database
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 168 texts...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches: 100%|██████████| 6/6 [00:10<00:00,  1.76s/it]


Generated embeddings with shape: (168, 384)
Adding 168 documents to vector store...
Successfully added 168 documents to vector store
Total documents in collection: 504


### Retriever Pipeline From VectorStore

In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [14]:
rag_retriever

In [15]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.55it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [16]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.71it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [18]:
rag_retriever.retrieve("Knowledge Graph Construction")

Retrieving documents for query: 'Knowledge Graph Construction'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.91it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_735038de_4',
  'content': 'Knowledge graphs (KGs) (Auer et al., 2007;\nJi et al., 2022), as structured abstractions of real-\nworld entities and their relations, can be expected\nto effectively supplement existing semantic-based\nRAG approaches by integrating structured factual\nknowledge. Knowledge within a KG, represented\nin the form of triplets (head entity, relation, tail\nentity), is naturally linked through overlapping en-\ntities. A simpliﬁed workﬂow for utilizing KGs in\nRAG is shown in Fig. 1, where relevant triplets are\nretrieved to augment the context for response gen-\narXiv:2502.06864v1  [cs.CL]  8 Feb 2025',
  'metadata': {'keywords': '',
   'file_type': 'pdf',
   'total_pages': 13,
   'page': 0,
   'subject': '',
   'creator': '',
   'file_path': '..\\data\\text_files\\pdf\\RAG_KG1. draft1.pdf',
   'creationdate': "D:20251204083830Z00'00'",
   'source': '..\\data\\text_files\\pdf\\RAG_KG1. draft1.pdf',
   'moddate': "D:20251204083830Z00'00'",
   'format': 

### Integration Vectordb Context pipeline With LLM output

In [21]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")


llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [23]:
answer=rag_simple("What is Knowledge Graph?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Knowledge Graph?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.94it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


A Knowledge Graph is a structured representation of real‑world entities and the relationships between them, typically modeled as triples (head entity, relation, tail entity).


### Enhanced RAG Pipeline Features

In [26]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Knowledge Graph?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Knowledge Graph?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.93it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: A Knowledge Graph is a structured representation of real‑world entities and the relationships between them, typically expressed as triplets (head entity, relation, tail entity) that are interconnected through shared entities.
Sources: [{'source': 'RAG_KG1. draft1.pdf', 'page': 0, 'score': 0.19153064489364624, 'preview': 'Knowledge graphs (KGs) (Auer et al., 2007;\nJi et al., 2022), as structured abstractions of real-\nworld entities and their relations, can be expected\nto effectively supplement existing semantic-based\nRAG approaches by integrating structured factual\nknowledge. Knowledge within a KG, represented\nin the...'}, {'source': 'RAG_KG1. draft1.pdf', 'page': 0, 'score': 0.19153064489364624, 'preview': 'Knowledge graphs (KGs) (Auer et al., 2007;\nJi et al., 2022), as structured abstractions of real-\nworld entities and their relations, can be expected\nto effectively supplement existing semantic-based\nRAG approaches by integrating structured factual\nknowledge. Knowl

In [28]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is Knowledge Graph?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What is Knowledge Graph?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 47.95it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Knowledge graphs (KGs) (Auer et al., 2007;
Ji et al., 2022), as structured abstractions of real-
world entities and their relations, can be expected
to effectively supplement existing semantic-based
RAG approaches by integrating structured factual
kno

wledge. Knowledge within a KG, represented
in the form of triplets (head entity, relation, tail
entity), is naturally linked through overlapping en-
tities. A simpliﬁed workﬂow for utilizing KGs in
RAG is shown in Fig. 1, where relevant triplets are
retrieved to augment the context for response gen-
arXiv:2502.06864v1  [cs.CL]  8 Feb 2025

Knowledge graphs (KGs) (Auer et al., 2007;
Ji et al., 2022), as structured abstractions of real-
world entities and their relations, can be expected
to effectively supplement existing semantic-based
RAG approaches by integrating structured factual
knowledge. Knowledge within a KG, represented
in the form of triplets (head entity, relation, tail
entity), is naturally linked through overlapping en-
tities. A simpliﬁed workﬂow for utilizing KGs in
RAG is shown in Fig. 1, where relevant triplets are
retrieved to augment the context for response gen-
arXiv:2502.06864v1  [cs.CL]  8 Feb 2025

Knowledge graphs (KGs) (Auer et al., 2007;
Ji et al., 2022), as s